In [ ]:
from functools import partial

import numpy as np
import pandas as pd
import tensorflow as tf
from bayesflow.amortizers import AmortizedPosterior
from bayesflow.networks import InvertibleNetwork
from bayesflow.simulation import GenerativeModel, Prior, Simulator
from bayesflow.trainers import Trainer
from tensorflow.keras.layers import Dense, GRU, LSTM

from helper_functions import batch_uniform_prior, normalize_household_data

In [ ]:
# 12 params
lb = np.ones(12) * 0 
ub = np.ones(12) 
prior_mean = (ub + lb) / 2
# standard deviation of uniform
prior_std = (ub - lb) / np.sqrt(12)
param_names = [f'theta_{i}' for i in range(12)]
n_params = len(param_names)

In [ ]:
df = pd.read_csv('data/simulated_data_formatted_alpha_rnd_1.txt',
                 delimiter=' ', index_col=0)    

minimal_length = 7
df_norm = normalize_household_data(df, return_list=True, minimal_length=minimal_length)
n_households = len(df_norm)
print(n_households)

def batch_simulator(param_batch: np.ndarray) -> np.ndarray:    
    # sample from dataframe
    n_samples = param_batch.shape[0]    
    sim_data = np.zeros((n_samples, n_households, minimal_length, df_norm.shape[-1]))
    for i in range(n_samples):
        idx = np.random.choice(n_households, n_households, replace=True)
        # now return only rows where household id is in idx
        sim_data[i, :] = df_norm[idx]
    return sim_data

In [ ]:
bayesflow_prior = Prior(batch_prior_fun=partial(batch_uniform_prior, lb=lb, ub=ub),
                            param_names=param_names)
bayes_simulator = Simulator(batch_simulator_fun=batch_simulator)
generative_model = GenerativeModel(prior=bayesflow_prior, simulator=bayes_simulator,
                                  name="Normalizing Flow Generative Model")

In [ ]:
def configurator(forward_dict: dict) -> dict:
    out_dict = {}

    # Extract data
    x = forward_dict["sim_data"].astype(np.float32)    
    # x = (x - x_mean) / x_std, already normalized
    #idx_keep = np.all(np.isfinite(x), axis=(1, 2))
    out_dict['summary_conditions'] = x#[idx_keep]
    
    # if simulations contains nan
    #if not np.all(idx_keep):
    #    print(f'Invalid value(s) encountered...removing {idx_keep.size - np.sum(idx_keep)} entry(ies) from batch')
    
    # Extract params
    if 'parameters' in forward_dict.keys():
        forward_dict["prior_draws"] = forward_dict["parameters"]
    if 'prior_draws' in forward_dict.keys():
        params = forward_dict["prior_draws"].astype(np.float32)
        params = (params - prior_mean) / prior_std
        out_dict['parameters'] = params#[idx_keep]
    return out_dict

In [ ]:
class GroupSummaryNetwork(tf.keras.Model):
    
    def __init__(
        self, summary_dim=10, rnn_units=128, n_groups=128, use_lstm=False, **kwargs
    ):
        super().__init__(**kwargs)

        self.rnn = LSTM(rnn_units) if use_lstm else GRU(rnn_units)
        self.out_layer = Dense(summary_dim, activation="linear")
        self.summary_dim = summary_dim
        self.n_groups = n_groups
        
    def call(self, x, **kwargs):
        """Performs a forward pass through the network by first passing `x` through the same rnn network for
        each household and then pooling the outputs across households.

        Parameters
        ----------
        x : tf.Tensor
            Input of shape (batch_size, n_groups, n_time_steps, n_features)

        Returns
        -------
        out : tf.Tensor
            Output of shape (batch_size, summary_dim)
        """
        # iterate over groups
        out_list = []  # list to store outputs of LSTM for each group
        for i in range(self.n_groups):
            out = self.rnn(x[:, i], **kwargs)  # (batch_size, lstm_units)
            out_list.append(out)
        # one could apply a time-resolved equivariant layer here and then pool
        # max pooling over groups
        out = tf.reduce_max(out_list, axis=0)  # (batch_size, lstm_units)
        # apply dense layer
        out = self.out_layer(out, **kwargs)  # (batch_size, summary_dim)
        return out

In [ ]:
power_k_hidden_units = int(np.ceil(np.log2(df_norm.shape[1])))
summary_net = GroupSummaryNetwork(summary_dim=n_params*2,
                                  rnn_units=2**power_k_hidden_units)
inference_net = InvertibleNetwork(num_params=n_params,
                                  num_coupling_layers=7,
                                  coupling_design='spline',
                                  coupling_settings={
                                      "num_dense": 3,
                                      "dense_args": dict(
                                          activation='swish',
                                          kernel_regularizer=tf.keras.regularizers.l2(1e-4)
                                      )
                                  })

In [ ]:
amortizer = AmortizedPosterior(inference_net=inference_net, summary_net=summary_net)
checkpoint_path = 'amortizer-test'

# build the trainer with networks and generative model
trainer = Trainer(amortizer=amortizer,
                  configurator=configurator,
                  generative_model=generative_model,
                  checkpoint_path=checkpoint_path,
                  max_to_keep=1)

In [ ]:
history = trainer.train_online(epochs=2,
                               iterations_per_epoch=100,
                               batch_size=128,
                               early_stopping=False,
                               validation_sims=100)